# Итоговый проект: прогноз дефолта клиента

**Дисциплина:** Автоматизация процессов разработки и тестирования моделей машинного обучения

В работе использую датасет Default of Credit Card Clients. Моя задача — предсказать, будет ли дефолт клиента в следующем месяце.

Показываю подготовку данных, проверку Pandera, новые признаки, обучение моделей, MLflow, DVC, API и мониторинг.

## 1. Подготовка проекта

Подключаю библиотеки и указываю папку проекта. Перед запуском устанавливаю зависимости из `requirements.txt`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print("Папка проекта:", project_root)

## 2. Загрузка данных

Загружаю датасет UCI и открываю первые пять строк. Если файл уже есть, повторно его не скачиваю.

In [ ]:
from src.data.download import TARGET, download_dataset

raw_path = download_dataset()
data = pd.read_csv(raw_path)
print("Размер датасета:", data.shape)
data.head()

## 3. Небольшой анализ данных

Смотрю типы данных, пропуски и долю клиентов с дефолтом.

In [ ]:
print(data.dtypes.value_counts())
print("Всего пропусков:", data.isna().sum().sum())
print("Доля дефолтов:", round(data[TARGET].mean(), 3))
data[["LIMIT_BAL", "AGE", "PAY_0"]].describe()

Строю два графика: распределение целевой переменной и возраст клиентов.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
data[TARGET].value_counts().sort_index().plot.bar(ax=axes[0])
axes[0].set_title("Количество клиентов по классу")
axes[0].set_xlabel("Дефолт: 0 — нет, 1 — да")
data["AGE"].hist(bins=20, ax=axes[1])
axes[1].set_title("Возраст клиентов")
axes[1].set_xlabel("Возраст")
plt.tight_layout()
plt.show()

Смотрю, как доля дефолтов связана с количеством месяцев просрочки в последнем периоде.

In [ ]:
default_by_payment = data.groupby("PAY_0")[TARGET].mean()
default_by_payment.plot.bar(figsize=(9, 3))
plt.title("Доля дефолтов по PAY_0")
plt.ylabel("Доля дефолтов")
plt.show()

## 4. Валидация данных

Проверяю типы, пропуски и допустимые значения через Pandera.

In [ ]:
from src.data.validation import validate_data

validated_data = validate_data(data)
print("Проверка прошла успешно:", len(validated_data), "строк")

Для проверки специально делаю возраст одного клиента равным 150. Валидатор должен найти ошибку.

In [ ]:
wrong_data = data.copy()
wrong_data.loc[0, "AGE"] = 150

try:
    validate_data(wrong_data)
    print("Ошибка не найдена")
except Exception:
    print("Проверка сработала: возраст 150 не допускается")

## 5. Подготовка данных

Привожу редкие значения категорий к группе «другое» и делю данные на train и test. Долю дефолтов сохраняю с помощью stratify.

In [ ]:
from src.data.make_dataset import prepare_data

train_data, test_data = prepare_data()
print("Train:", train_data.shape)
print("Test:", test_data.shape)
print("Доля дефолтов train:", round(train_data[TARGET].mean(), 3))
print("Доля дефолтов test:", round(test_data[TARGET].mean(), 3))

## 6. Новые признаки

Считаю среднюю просрочку, число месяцев с задержкой, средние суммы счетов и оплат. Возраст делю на группы.

In [ ]:
from src.features.build_features import create_features

features = create_features(train_data.head(5))
features[[
    "payment_delay_mean", "payment_delay_count",
    "bill_amount_mean", "payment_amount_mean", "age_group"
]]

## 7. Sklearn Pipeline

Создаю пайплайн. Он сам заполняет пропуски, масштабирует числовые признаки и кодирует категории.

In [ ]:
from src.models.pipeline import create_pipeline

pipeline = create_pipeline("logistic", C=1.0)
print(pipeline.named_steps.keys())
print(pipeline.named_steps["classifier"])

## 8. Обучение и эксперименты

Запускаю пять вариантов моделей. Для каждого сохраняю параметры, метрики на validation, ROC-кривую и модель в MLflow. Затем подбираю параметры лучшего алгоритма через GridSearchCV и проверяю его на test.

In [ ]:
from src.models.train import run_training

experiment_results, final_metrics = run_training()

## 9. Сравнение экспериментов

Сортирую результаты по ROC-AUC на validation. Чем больше значение, тем лучше модель отделяет классы.

In [ ]:
display(experiment_results.round(3))
print("Лучший эксперимент:", experiment_results.iloc[0]["experiment"])

Показываю ROC-AUC для всех пяти экспериментов.

In [ ]:
experiment_results.sort_values("roc_auc").plot.barh(
    x="experiment", y="roc_auc", legend=False, figsize=(8, 4)
)
plt.xlabel("ROC-AUC на validation")
plt.ylabel("Эксперимент")
plt.show()

## 10. Качество итоговой модели

Вывожу ROC-AUC, Precision, Recall и F1 на отложенной тестовой выборке.

In [ ]:
metric_names = ["roc_auc", "precision", "recall", "f1"]
for metric_name in metric_names:
    print(metric_name, ":", round(final_metrics[metric_name], 3))
print("Параметры:", final_metrics["best_params"])

Открываю ROC-кривую, которую сохранил скрипт обучения.

In [ ]:
roc_image = plt.imread("reports/roc_curve.png")
plt.figure(figsize=(7, 5))
plt.imshow(roc_image)
plt.axis("off")
plt.show()

## 11. MLflow

Сохраняю пять отдельных экспериментов и итоговый запуск. Для просмотра запускаю из терминала:

```bash
mlflow ui --backend-store-uri ./mlruns --port 5000
```

В браузере открываю **http://localhost:5000** и сравниваю метрики и параметры.

## 12. DVC

Пайплайн состоит из двух стадий: `prepare` и `train`. Для воспроизведения запускаю:

```bash
dvc add data/raw/credit_default.csv
dvc repro
```

Открываю описание стадий из файла `dvc.yaml`.

In [ ]:
print(Path("dvc.yaml").read_text(encoding="utf-8"))

## 13. Автоматические тесты

Запускаю pytest для проверки подготовки признаков, метрик, валидации и API.

In [ ]:
import subprocess

subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 14. Проверка FastAPI

Отправляю одну строку тестовой выборки в `/predict`. Здесь использую тестовый клиент FastAPI, поэтому отдельный сервер пока не нужен.

In [ ]:
from fastapi.testclient import TestClient

from src.api.app import app
from src.features.build_features import RAW_FEATURES

client = TestClient(app)
client_row = test_data.iloc[0][RAW_FEATURES].to_dict()
response = client.post("/predict", json=client_row)
print("HTTP:", response.status_code)
print(response.json())

## 15. Docker

Собираю образ и запускаю API через скрипт:

```bash
bash scripts/run_docker.sh
```

Проверяю `/docs` по адресу **http://localhost:8000/docs**.

## 16. Простой мониторинг

Сравниваю вероятности дефолта на train и test через PSI. Дополнительно в проекте есть скрипт, который отправляет тестовые данные в настоящий HTTP API.

In [ ]:
import joblib

from src.monitoring.drift import calculate_psi

trained_model = joblib.load("models/credit_default_model.joblib")
reference = pd.read_csv("reports/reference_probabilities.csv")["probability"]
current = trained_model.predict_proba(test_data.drop(columns=[TARGET]))[:, 1]
psi = calculate_psi(reference, current)
print("PSI:", round(psi, 4))

Для проверки через работающий API запускаю в терминале:

```bash
python -m src.monitoring.drift --sample-size 100
```

## 17. Вывод

Проверяю данные, создаю новые признаки и обучаю пять вариантов модели. Потом выбираю лучший вариант, подбираю параметры и оцениваю результат на test.

Для автоматизации использую Pandera, MLflow, DVC, pytest, GitHub Actions, Docker и FastAPI. В проекте также есть простой расчёт PSI.